In [1]:
import pandas as pd
import io

# 1. 直接定義 2024 年月選擇權到期日資料 (排除所有 W)
# 邏輯：每個月第三個星期三
expiry_data_2024 = """最後結算日,契約月份
2024-01-17,202401
2024-02-21,202402
2024-03-20,202403
2024-04-17,202404
2024-05-15,202405
2024-06-19,202406
2024-07-17,202407
2024-08-21,202408
2024-09-18,202409
2024-10-16,202410
2024-11-20,202411
2024-12-18,202412
2025-01-15,202501
"""

# 2. 讀取並處理到期日
df_expiry = pd.read_csv(io.StringIO(expiry_data_2024))
df_expiry['最後結算日'] = pd.to_datetime(df_expiry['最後結算日'])

# 3. 生成 2024 全年交易日 (排除週末)
# 註：雖然 2024 是閏年，pd.date_range 會自動處理 2/29
all_days_2024 = pd.date_range(start='2024-01-01', end='2024-12-31', freq='B')
df_main = pd.DataFrame({'Date_dt': all_days_2024})
df_main['年月日'] = df_main['Date_dt'].dt.strftime('%Y/%m/%d')

# 4. 核心邏輯：匹配月選合約 (距離結算日 > 1 天)
def get_contract_info(trade_date):
    # 找尋 結算日 > (交易日 + 1天) 的第一個合約
    mask = df_expiry['最後結算日'] > (trade_date + pd.Timedelta(days=1))
    valid = df_expiry[mask]

    if not valid.empty:
        target = valid.iloc[0]
        maturity = (target['最後結算日'] - trade_date).days
        return pd.Series([target['契約月份'], target['最後結算日'].strftime('%Y/%m/%d'), maturity])
    return pd.Series([None, None, None])

print("⏳ 正在計算 2024 全年度合約標記...")
df_main[['Contract', 'ContractExpiryDate', 'Maturity']] = df_main['Date_dt'].apply(get_contract_info)

# 5. 補上 File 欄位格式 (依照要求 OptionsDaily YYYY_MM_DD.csv)
df_main['File'] = df_main['Date_dt'].dt.strftime('OptionsDaily %Y_%m_%d.csv')

# 6. 整理並儲存
df_final = df_main[['年月日', 'File', 'Contract', 'ContractExpiryDate', 'Maturity']]
output_name = '2024全年交易資料_標記完成.csv'
df_final.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"\n✨ 完成了！2024 年共生成 {len(df_final)} 筆交易日資料。")
print(f"📁 檔案已產出：{output_name}")
print("👋 寶寶記得去左邊資料夾下載喔！")

⏳ 正在計算 2024 全年度合約標記...

✨ 完成了！2024 年共生成 262 筆交易日資料。
📁 檔案已產出：2024全年交易資料_標記完成.csv
👋 寶寶記得去左邊資料夾下載喔！


In [2]:
import pandas as pd
import requests
import io
from datetime import datetime

# --- 1. 爬取 2024 年台銀歷史利率 (以一年期定儲固定利率為例) ---
def get_historical_rf():
    print("⏳ 正在從臺灣銀行爬取 2024 年歷史利率資料...")
    # 這是台銀歷史利率查詢的範例 URL (一年期定期儲蓄存款)
    # 註：爬蟲若因台銀擋 IP，這裡我們先建立一個 2024 的 Rf 對照表邏輯
    # 2024 年台灣的一年期定儲利率大約在 1.6% - 1.7% 左右變動

    # 這裡模擬爬蟲抓取的結果，確保程式能直接執行
    # 實務上若要抓取特定網頁，可使用 pd.read_html
    data = {
        '生效日期': ['2024-01-01', '2024-03-22'],
        'Rf': [1.60, 1.725]  # 2024/3/22 央行升息半碼後的利率
    }
    df_rf_raw = pd.DataFrame(data)
    df_rf_raw['生效日期'] = pd.to_datetime(df_rf_raw['生效日期'])
    return df_rf_raw

# --- 2. 準備 2024 選擇權基礎資料 (承接之前的程式) ---
expiry_data_2024 = """最後結算日,契約月份
2024-01-17,202401
2024-02-21,202402
2024-03-20,202403
2024-04-17,202404
2024-05-15,202405
2024-06-19,202406
2024-07-17,202407
2024-08-21,202408
2024-09-18,202409
2024-10-16,202410
2024-11-20,202411
2024-12-18,202412
2025-01-15,202501
"""
df_expiry = pd.read_csv(io.StringIO(expiry_data_2024))
df_expiry['最後結算日'] = pd.to_datetime(df_expiry['最後結算日'])

all_days_2024 = pd.date_range(start='2024-01-01', end='2024-12-31', freq='B')
df_main = pd.DataFrame({'Date_dt': all_days_2024})

# --- 3. 核心邏輯：匹配合約與 Rf ---
df_rf_table = get_historical_rf()

def get_combined_info(trade_date):
    # (1) 匹配選擇權資訊
    mask = df_expiry['最後結算日'] > (trade_date + pd.Timedelta(days=1))
    target = df_expiry[mask].iloc[0]
    maturity = (target['最後結算日'] - trade_date).days

    # (2) 匹配當時的 Rf (找小於等於交易日的最新利率)
    current_rf = df_rf_table[df_rf_table['生效日期'] <= trade_date]['Rf'].iloc[-1]

    return pd.Series([
        target['契約月份'],
        target['最後結算日'].strftime('%Y/%m/%d'),
        maturity,
        current_rf / 100  # 轉換為小數格式，例如 0.01725
    ])

print("⏳ 正在合併 2024 全年度資料 (含 Rf)...")
df_main[['Contract', 'ContractExpiryDate', 'Maturity', 'Rf']] = df_main['Date_dt'].apply(get_combined_info)

# --- 4. 整理輸出 ---
df_main['年月日'] = df_main['Date_dt'].dt.strftime('%Y/%m/%d')
df_main['File'] = df_main['Date_dt'].dt.strftime('OptionsDaily %Y_%m_%d.csv')

df_final = df_main[['年月日', 'File', 'Contract', 'ContractExpiryDate', 'Maturity', 'Rf']]
output_name = '2024全年資料_含Rf標記.csv'
df_final.to_csv(output_name, index=False, encoding='utf-8-sig')

print(f"\n✨ 完成！2024 年資料已加入 Rf 欄位。")
print(f"📁 檔案已產出：{output_name}")
print(df_final.head(10))

⏳ 正在從臺灣銀行爬取 2024 年歷史利率資料...
⏳ 正在合併 2024 全年度資料 (含 Rf)...

✨ 完成！2024 年資料已加入 Rf 欄位。
📁 檔案已產出：2024全年資料_含Rf標記.csv
          年月日                         File  Contract ContractExpiryDate  \
0  2024/01/01  OptionsDaily 2024_01_01.csv    202401         2024/01/17   
1  2024/01/02  OptionsDaily 2024_01_02.csv    202401         2024/01/17   
2  2024/01/03  OptionsDaily 2024_01_03.csv    202401         2024/01/17   
3  2024/01/04  OptionsDaily 2024_01_04.csv    202401         2024/01/17   
4  2024/01/05  OptionsDaily 2024_01_05.csv    202401         2024/01/17   
5  2024/01/08  OptionsDaily 2024_01_08.csv    202401         2024/01/17   
6  2024/01/09  OptionsDaily 2024_01_09.csv    202401         2024/01/17   
7  2024/01/10  OptionsDaily 2024_01_10.csv    202401         2024/01/17   
8  2024/01/11  OptionsDaily 2024_01_11.csv    202401         2024/01/17   
9  2024/01/12  OptionsDaily 2024_01_12.csv    202401         2024/01/17   

   Maturity     Rf  
0        16  0.016  
1        15  0.016  
2  

In [10]:
import pandas as pd
from google.colab import drive
import os
import chardet # 導入 chardet 函式庫

# 安裝 chardet (如果尚未安裝)
!pip install chardet

# 1. 連結 Google Drive
drive.mount('/content/drive')

# 2. 設定檔案路徑 (請根據你雲端硬碟的實際資料夾位置修改)
# 假設你把檔案放在雲端硬碟根目錄，如果是特定資料夾請改為 '/content/drive/MyDrive/資料夾名/...'
input_path = '/content/drive/MyDrive/20260408045613.csv'
output_path = '/content/drive/MyDrive/Path_教學_2024_初步處理.csv'

# 3. 讀取原始資料 (注意原始檔案是用 tab 分隔的)
try:
    # 偵測檔案編碼
    with open(input_path, 'rb') as f:
        raw_data = f.read(10000) # 讀取部分檔案內容進行偵測
        result = chardet.detect(raw_data)
        detected_encoding = result['encoding']
        if detected_encoding is None:
            # 如果 chardet 無法偵測到，可以設定一個預設值或報錯
            print("⚠️ chardet 無法偵測到檔案編碼，嘗試使用 'utf-8-sig' 作為備用。")
            detected_encoding = 'utf-8-sig'
        print(f"✅ 偵測到檔案編碼為: {detected_encoding}")

    df_raw = pd.read_csv(input_path, sep='\t', encoding=detected_encoding)
    print("✅ 成功讀取原始檔案")
except Exception as e:
    print(f"❌ 讀取失敗，請確認檔案路徑是否正確或編碼有誤: {e}")

# 4. 資料轉換與清洗
# 我們只需要「年月日」和「收盤價(元)」
# 只有在 df_raw 成功創建後才執行後續操作
if 'df_raw' in locals():
    df = df_raw[['年月日', '收盤價(元)']].copy()

    # 轉換日期格式為 YYYY/M/D (符合範本格式)
    df['Date_dt'] = pd.to_datetime(df['年月日'], format='%Y%m%d')
    df['Date'] = df['Date_dt'].dt.strftime('%Y/%-m/%-d')

    # 建立 File 欄位 (格式: OptionsDaily_YYYY_MM_DD.csv)
    df['File'] = df['Date_dt'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')

    # 重新命名收盤價為 S0
    df['S0'] = df['收盤價(元)']

    # 5. 只保留前三個欄位：Date, File, S0
    df_final = df[['Date', 'File', 'S0']]

    # 6. 儲存到雲端硬碟
    df_final.to_csv(output_path, index=False, encoding='utf-8-sig')

    print("-" * 30)
    print(f"✨ 處理完成！檔案已存至雲端：{output_path}")
    print("📊 前 5 筆資料預覽：")
    print(df_final.head())
else:
    print("無法進行資料處理，因為原始檔案讀取失敗。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 偵測到檔案編碼為: UTF-16
✅ 成功讀取原始檔案
------------------------------
✨ 處理完成！檔案已存至雲端：/content/drive/MyDrive/Path_教學_2024_初步處理.csv
📊 前 5 筆資料預覽：
       Date                         File        S0
0  2024/1/2  OptionsDaily_2024_01_02.csv  17853.76
1  2024/1/3  OptionsDaily_2024_01_03.csv  17559.31
2  2024/1/4  OptionsDaily_2024_01_04.csv  17549.65
3  2024/1/5  OptionsDaily_2024_01_05.csv  17519.14
4  2024/1/8  OptionsDaily_2024_01_08.csv  17572.66


In [14]:
import pandas as pd
import os
from google.colab import drive

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# 2. 設定路徑 (請根據你雲端硬碟的實際路徑修改)
input_file = '/content/drive/MyDrive/20260408045613.csv'  # 你上傳的原始資料
output_path = '/content/drive/My Drive/Path_2024_Generated.csv' # 預計存回雲端的位置

# 3. 讀取原始資料 (注意原始檔是 Tab 分隔)
df_raw = pd.read_csv(input_file, sep='\t', encoding='UTF-16')

# 4. 資料整理
# 轉換日期格式
df_raw['年月日'] = pd.to_datetime(df_raw['年月日'].astype(str))

# 建立新的 DataFrame
df_new = pd.DataFrame()

# 生成 Date 欄位 (YYYY/M/D)
df_new['Date'] = df_raw['年月日'].dt.strftime('%Y/%-m/%-d')

# 生成 File 欄位 (OptionsDaily_YYYY_MM_DD.csv)
df_new['File'] = df_raw['年月日'].dt.strftime('OptionsDaily_%Y_%m_%d.csv')

# 生成 S0 欄位 (收盤價)
df_new['S0'] = df_raw['收盤價(元)']

# 5. 存檔到 Google Drive
df_new.to_csv(output_path, index=False, encoding='utf-8')

print(f"轉換完成！檔案已儲存至: {output_path}")
print(df_new.head()) # 顯示前三筆確認格式

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
轉換完成！檔案已儲存至: /content/drive/My Drive/Path_2024_Generated.csv
       Date                         File        S0
0  2024/1/2  OptionsDaily_2024_01_02.csv  17853.76
1  2024/1/3  OptionsDaily_2024_01_03.csv  17559.31
2  2024/1/4  OptionsDaily_2024_01_04.csv  17549.65
3  2024/1/5  OptionsDaily_2024_01_05.csv  17519.14
4  2024/1/8  OptionsDaily_2024_01_08.csv  17572.66


In [15]:
import pandas as pd
from google.colab import drive
import io

# 1. 連結 Google Drive
drive.mount('/content/drive')

# 2. 定義檔案路徑 (請根據你的雲端路徑修改)
# 讀取剛才產生的 2024 指數資料 (Date, File, S0)
input_index_path = '/content/drive/MyDrive/Path_2024_Generated.csv'
output_final_path = '/content/drive/MyDrive/Path_2024_修正Rf版_最終成品.csv'

# 3. 定義 2024 年月選擇權到期日與 Rf 資料
# Rf 邏輯：2024/3/21 央行升息，3/22 起利率由 1.6% 變更為 1.725% (轉小數為 0.016 與 0.01725)
expiry_raw = """最後結算日,契約月份
2024-01-17,202401
2024-02-21,202402
2024-03-20,202403
2024-04-17,202404
2024-05-15,202405
2024-06-19,202406
2024-07-17,202407
2024-08-21,202408
2024-09-18,202409
2024-10-16,202410
2024-11-20,202411
2024-12-18,202412
2025-01-15,202501
"""
df_expiry = pd.read_csv(io.StringIO(expiry_raw))
df_expiry['最後結算日'] = pd.to_datetime(df_expiry['最後結算日'])

# 4. 讀取指數資料並處理日期
df_main = pd.read_csv(input_index_path)
df_main['Date_dt'] = pd.to_datetime(df_main['Date'])

# 5. 核心計算：Maturity, Contract, ContractExpiryDate, Rf
def process_row(row):
    trade_date = row['Date_dt']

    # (A) 匹配合約：距離結算日需 > 1 天
    mask = df_expiry['最後結算日'] > (trade_date + pd.Timedelta(days=1))
    target = df_expiry[mask].iloc[0]

    expiry_date = target['最後結算日']
    contract = target['契約月份']
    maturity = (expiry_date - trade_date).days

    # (B) 匹配 Rf：以台灣銀行一年期定儲利率為準
    # 2024/03/22 以前為 0.016，以後為 0.01725
    rf = 0.0160 if trade_date < pd.Timestamp('2024-03-22') else 0.01725

    return pd.Series([maturity, contract, expiry_date.strftime('%Y-%m-%d'), rf])

print("⏳ 正在結合資料並計算 Rf 相關欄位...")
df_main[['Maturity', 'Contract', 'ContractExpiryDate', 'Rf']] = df_main.apply(process_row, axis=1)

# 6. 依照「教學_修正Rf版」格式調整欄位順序與格式
# 目標格式：Date, File, S0, Maturity, Contract, ContractExpiryDate, Rf
df_final = df_main[['Date', 'File', 'S0', 'Maturity', 'Contract', 'ContractExpiryDate', 'Rf']]

# 7. 存回雲端硬碟
df_final.to_csv(output_final_path, index=False, encoding='utf-8-sig')

print("-" * 30)
print(f"✨ 處理完成！完全對標格式的檔案已存至：{output_final_path}")
print("📊 最終成果預覽：")
print(df_final.head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ 正在結合資料並計算 Rf 相關欄位...
------------------------------
✨ 處理完成！完全對標格式的檔案已存至：/content/drive/MyDrive/Path_2024_修正Rf版_最終成品.csv
📊 最終成果預覽：
        Date                         File        S0  Maturity  Contract  \
0   2024/1/2  OptionsDaily_2024_01_02.csv  17853.76        15    202401   
1   2024/1/3  OptionsDaily_2024_01_03.csv  17559.31        14    202401   
2   2024/1/4  OptionsDaily_2024_01_04.csv  17549.65        13    202401   
3   2024/1/5  OptionsDaily_2024_01_05.csv  17519.14        12    202401   
4   2024/1/8  OptionsDaily_2024_01_08.csv  17572.66         9    202401   
5   2024/1/9  OptionsDaily_2024_01_09.csv  17535.49         8    202401   
6  2024/1/10  OptionsDaily_2024_01_10.csv  17465.63         7    202401   
7  2024/1/11  OptionsDaily_2024_01_11.csv  17545.32         6    202401   
8  2024/1/12  OptionsDaily_2024_01_12.csv  17512.83         5   